# Demo: evaluación de modelos con el Titanic

Píldora **Evaluación de modelos y métricas de rendimiento** · M2 Data Analytics & Machine Learning · 11/11/2026

Usamos el mismo dataset del taller de EDA. Ahora damos el paso siguiente: entrenar un modelo que predice quién sobrevive y **medir cómo de bueno es**.

| Paso | Qué hacemos | Tiempo |
|---|---|---|
| 1 | Cargar y preparar los datos | 2 min |
| 2 | Separar train y test, y entrenar | 3 min |
| 3 | Matriz de confusión y accuracy | 3 min |
| 4 | Precision, recall y F1 | 3 min |
| 5 | El modelo tonto «todos mueren» | 2 min |
| 6 | Mover el umbral | 2 min |
| 7 | Curva ROC y AUC | 2 min |
| Extra | Métricas de regresión (si sobra tiempo) | — |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
    precision_score, recall_score, f1_score, classification_report,
    roc_auc_score, RocCurveDisplay,
    mean_absolute_error, mean_squared_error, r2_score,
)

## Paso 1 · Cargar y preparar los datos

Lo mismo que en el EDA: quitamos columnas que no ayudan, rellenamos nulos (mediana para la edad, moda para el puerto de embarque) y convertimos el texto en números para que el modelo lo entienda.

Convertir `Sex` y `Embarked` en números se llama **encoding**: mañana (12/11) lo veréis a fondo en la lección de categorical encoding.

In [ ]:
URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(URL)

df = df[["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
df = pd.get_dummies(df, columns=["Embarked"], drop_first=True, dtype=int)

print(df.shape)
df.head()

## Paso 2 · Separar train y test, y entrenar

Como vimos ayer: el modelo aprende con **train** y lo evaluamos con **test**, pasajeros que nunca ha visto. Usamos `stratify=y` para que en los dos grupos haya la misma proporción de supervivientes.

In [ ]:
X = df.drop(columns="Survived")
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)

print("Pasajeros de train:", len(X_train))
print("Pasajeros de test: ", len(X_test))

## Paso 3 · Matriz de confusión y accuracy

**Ojo:** scikit-learn pone primero la clase 0 («muere»). En la matriz, las filas son la realidad y las columnas la predicción, así que el orden es:

| | Predijo: muere | Predijo: sobrevive |
|---|---|---|
| **Murió** | TN | FP |
| **Sobrevivió** | FN | TP |

In [ ]:
y_pred = modelo.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Muere", "Sobrevive"]).plot(cmap="Blues")
plt.title("Matriz de confusión")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TP={tp}  FN={fn}  FP={fp}  TN={tn}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")

## Paso 4 · Precision, recall y F1

Pregunta al público antes de ejecutar: con la matriz de arriba, ¿cuánto saldrá la precision? ¿Y el recall?

In [ ]:
print(f"Precision: {precision_score(y_test, y_pred):.2f}")
print(f"Recall:    {recall_score(y_test, y_pred):.2f}")
print(f"F1:        {f1_score(y_test, y_pred):.2f}")
print()
print(classification_report(y_test, y_pred, target_names=["Muere", "Sobrevive"]))

## Paso 5 · El modelo tonto «todos mueren»

Un `DummyClassifier` no mira los datos: siempre predice la clase más frecuente. Si nuestro modelo no le gana con claridad, no sirve.

In [ ]:
tonto = DummyClassifier(strategy="most_frequent")
tonto.fit(X_train, y_train)
y_tonto = tonto.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_tonto):.2f}   <- parece aceptable...")
print(f"Recall:   {recall_score(y_test, y_tonto):.2f}   <- no encuentra a ningún superviviente")
print(f"F1:       {f1_score(y_test, y_tonto, zero_division=0):.2f}")

## Paso 6 · Mover el umbral: precision contra recall

El modelo da una **probabilidad** de sobrevivir. Por defecto, por encima de 0,5 dice «sobrevive». Si movemos ese umbral, cambian precision y recall.

In [ ]:
y_proba = modelo.predict_proba(X_test)[:, 1]

filas = []
for umbral in [0.2, 0.35, 0.5, 0.65, 0.8]:
    pred = (y_proba >= umbral).astype(int)
    filas.append({
        "umbral": umbral,
        "precision": round(precision_score(y_test, pred, zero_division=0), 2),
        "recall": round(recall_score(y_test, pred), 2),
        "f1": round(f1_score(y_test, pred), 2),
    })

pd.DataFrame(filas)

## Paso 7 · Curva ROC y AUC

La curva ROC prueba todos los umbrales a la vez. La diagonal es un modelo que tira una moneda; cuanto más arriba a la izquierda, mejor.

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_proba, name="Regresión logística")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Azar (AUC = 0,5)")
plt.legend()
plt.title("Curva ROC")
plt.show()

print(f"AUC: {roc_auc_score(y_test, y_proba):.2f}")

## Extra · Métricas de regresión (si sobra tiempo)

Cambiamos de problema: predecimos el **precio del billete** (`Fare`) en vez de si sobrevive.

In [ ]:
Xr = df.drop(columns=["Fare", "Survived"])
yr = df["Fare"]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=42)

reg = LinearRegression().fit(Xr_train, yr_train)
yr_pred = reg.predict(Xr_test)

print(f"MAE:  {mean_absolute_error(yr_test, yr_pred):.2f} £")
print(f"RMSE: {mean_squared_error(yr_test, yr_pred) ** 0.5:.2f} £")
print(f"R²:   {r2_score(yr_test, yr_pred):.2f}")

## Conclusiones

1. **El accuracy puede mentir:** compáralo siempre con un modelo tonto.
2. **¿Qué error sale más caro?** Esa pregunta decide entre precision y recall.
3. **No hay «mejor métrica»:** hay la adecuada para tu problema.